# Grazing-Incidence Soft X-ray Spectrograph Tutorial

Soft X-rays (here, $\lambda = 15$–$20$ Å, $\sim 0.6$–$0.8$ keV) reflect efficiently
only at *grazing* incidence, so an imaging soft X-ray telescope cannot use a
normal-incidence parabola.  Instead it uses a **Wolter Type-I** optic: a
**paraboloid** followed by a confocal **hyperboloid**, both struck at a shallow
grazing angle.  The two reflections satisfy the Abbe sine condition far better
than a single mirror, giving good off-axis imaging.

This tutorial designs a slitless grazing-incidence spectrograph end-to-end in
[`optika`](https://optika.readthedocs.io):

* a **Wolter-I telescope** (paraboloid + hyperboloid),
* a **5000 line/mm transmission grating** placed in the converging beam, which
  disperses the first-order spectrum across the detector while the un-diffracted
  zero order forms a direct image, and
* a **2k $\times$ 4k CMOS detector** (10 µm pixels).

The design is driven by two numbers: a **spatial plate scale of 1 arcsec/pixel**
and a **first-order spectral range of 15–20 Å**.  Neither a grazing-incidence
telescope nor a transmission grating has been built in `optika` before, so this
is also a test of the package.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import astropy.units as u
import astropy.visualization
import named_arrays as na
import optika

## Focal length from the plate scale

The plate scale (angle subtended by one pixel) of a focusing system is
$p = w_\mathrm{pix} / f$.  Solving for the focal length with a
$1''$/pixel scale and 10 µm pixels:

In [ ]:
width_pixel = 10 * u.um
plate_scale = 1 * u.arcsec

focal_length = (width_pixel / plate_scale).to(
    u.mm, equivalencies=u.dimensionless_angles()
)
focal_length.to(u.m)

## Aperture and grazing angle

The collecting aperture is a design choice (here a 144 mm-diameter annulus).
Once the aperture radius $r_0$ and focal length $f$ are fixed, the grazing angle
follows from the Wolter-I geometry: a ray entering at radius $r_0$ must be bent
by a total angle $4\alpha$ (twice $2\alpha$, once at each mirror) to reach the
focus, so

$$r_0 = f \tan(4\alpha) \quad\Longrightarrow\quad
  \alpha = \tfrac14 \arctan(r_0 / f).$$

In [ ]:
radius_aperture = 72 * u.mm

grazing_angle = 0.25 * np.arctan(
    (radius_aperture / focal_length).to_value(u.dimensionless_unscaled)
) * u.rad
grazing_angle.to(u.deg)

A half-degree grazing angle is comfortably below the critical angle for
total external reflection of gold at these wavelengths (computed in the
*Reflectivity* section below), and it keeps the beam slow ($\approx f/14$),
which — as we will see — matters for the spectral resolution.

## Paraboloid (primary)

`optika` represents a parabola of revolution with
`optika.sags.ParabolicSag`, whose vertex sits at the local origin and whose
focus lies a distance equal to the `focal_length` argument from that vertex.

For grazing incidence we illuminate the parabola far from its vertex, on its
steep, nearly-cylindrical flank.  A ray parallel to the axis at radius $r_0$ is
deflected by $2\alpha$ toward the parabola focus a distance
$F_\mathrm{p} = r_0 / \tan(2\alpha)$ downstream.  The corresponding parabola
parameter is $f_\mathrm{p} = r_0^2 / (4 F_\mathrm{p})$ — here only a fraction of
a millimetre, since the illuminated flank is so steep.

We use a negative `focal_length` so the parabola opens toward $-z$ and the
$+z$-travelling rays reflect *inward*, and we translate it so the illuminated
annulus sits near $z = 0$ and the parabola focus lands at $z = F_\mathrm{p}$.

The aperture is an `optika.apertures.AnnularAperture` — a grazing mirror is
physically a thin **ring** with an empty centre, not a full disk.  A mirror of
axial length $L$ spans $\Delta r = L\tan\alpha$ in radius, so the ring half-width
is $L\tan\alpha/2$.  Because the grazing flank is so steep, even a short mirror is
*axially* long ($\sim 0.2$ m here), so we keep $L$ small and place the secondary
well downstream so the two mirrors do not overlap in $z$ — otherwise some rays
could not reach the secondary while travelling forward.

In [ ]:
focus_parabola = (radius_aperture / np.tan(2 * grazing_angle)).to(u.mm)
f_parabola = (radius_aperture**2 / (4 * focus_parabola)).to(u.mm)
print(f"parabola focus  F_p = {focus_parabola.to(u.mm):.1f}")
print(f"parabola param  f_p = {f_parabola.to(u.um):.1f}")

# A grazing mirror is a thin *annulus* (a ring with a central hole), not a full
# disk.  A mirror of axial length L spans Δr = L·tan(α) in radius, so the ring
# half-width is L·tan(α)/2.  Because the grazing flank is so steep, even a short
# mirror is axially long; we keep it short so the primary and secondary do not
# overlap in z (see the hyperboloid placement below).
mirror_length = 200 * u.mm
radial_halfwidth = (mirror_length * np.tan(grazing_angle) / 2).to(u.mm)
clearance = radial_halfwidth + 10 * u.mm

paraboloid = optika.surfaces.Surface(
    name="paraboloid",
    sag=optika.sags.ParabolicSag(focal_length=-f_parabola),
    material=optika.materials.Mirror(),
    aperture=optika.apertures.AnnularAperture(
        radius_inner=radius_aperture - clearance,
        radius_outer=radius_aperture + clearance,
    ),
    transformation=na.transformations.Cartesian3dTranslation(z=focus_parabola),
    is_pupil_stop=True,
)

## Hyperboloid (secondary)

The hyperboloid is **confocal** with the paraboloid: one of its two foci
coincides with the parabola focus $F_\mathrm{p}$ (its *far* focus), and the
other is the real system focus $F_\mathrm{sys}$, where the detector sits (its
*near* focus).  A conic mirror images one focus perfectly onto the other, so the
converging cone aimed at $F_\mathrm{p}$ is re-imaged to a perfect point at
$F_\mathrm{sys}$ — on-axis the Wolter-I has *zero* geometric aberration.

A confocal conic is fixed by its two foci plus one point it passes through.  We
require the hyperboloid to intercept the converging beam at an axial position
`z_intercept`; the beam radius there sets the third constraint.  From the foci
$z_N, z_F$ and the through-point we recover the standard conic parameters:
semi-major axis $a$ (from $2a = |d_F - d_N|$), eccentricity $e = c/a$ with
$c = (z_F - z_N)/2$, vertex radius of curvature $R = a(e^2 - 1)$, and conic
constant $k = -e^2 < -1$.

> **Note.** The parabola's true focus is one $f_\mathrm{p}$ *short* of its
> vertex translation, i.e. at $z = F_\mathrm{p} - f_\mathrm{p}$.  Using that
> exact value as the hyperboloid's far focus is what makes the on-axis image
> collapse to a point.

In [ ]:
z_system_focus = focal_length                          # detector / near focus
z_parabola_focus = focus_parabola - f_parabola         # exact parabola focus / far focus

z_center = 0.5 * (z_system_focus + z_parabola_focus)
c_hyp = 0.5 * (z_parabola_focus - z_system_focus)      # half focus separation = a*e

# Intercept the converging beam here.  The hyperboloid is also axially long, so
# we place it well downstream of the primary's z-extent: the two mirrors must
# not overlap in z, or part of the beam could not reach the secondary travelling
# forward.
z_intercept = 500 * u.mm
radius_intercept = radius_aperture * (z_parabola_focus - z_intercept) / z_parabola_focus

d_far = np.sqrt(radius_intercept**2 + (z_intercept - z_parabola_focus)**2)
d_near = np.sqrt(radius_intercept**2 + (z_intercept - z_system_focus)**2)

a_hyp = 0.5 * np.abs(d_far - d_near)
e_hyp = (c_hyp / a_hyp).to_value(u.dimensionless_unscaled)
radius_hyp = a_hyp * (e_hyp**2 - 1)
conic_hyp = -(e_hyp**2)
z_vertex_hyp = z_center - a_hyp

print(f"eccentricity e = {e_hyp:.6f}")
print(f"conic k        = {conic_hyp:.6f}")
print(f"vertex radius R = {radius_hyp.to(u.mm):.4f}")
print(f"vertex z        = {z_vertex_hyp.to(u.mm):.2f}")

hyperboloid = optika.surfaces.Surface(
    name="hyperboloid",
    sag=optika.sags.ConicSag(radius=-radius_hyp, conic=conic_hyp),
    material=optika.materials.Mirror(),
    aperture=optika.apertures.AnnularAperture(
        radius_inner=radius_intercept - clearance,
        radius_outer=radius_intercept + clearance,
    ),
    transformation=na.transformations.Cartesian3dTranslation(z=z_vertex_hyp),
)

## Transmission grating

A 5000 line/mm grating has a groove period $d = 200$ nm.  In first order the
diffraction angle is $\sin\beta = m\lambda/d$.  Placed a distance $\Lambda$
upstream of the focus, the grating shifts the first-order image laterally by
$\approx \Lambda\,\lambda/d$, giving a linear dispersion
$\mathrm{d}x/\mathrm{d}\lambda = \Lambda/d$.

We model it as a flat, **transmissive** surface (the default `Vacuum` material
passes light straight through) carrying a `ConstantRulingSpacing` whose `normal`
points along $x$ — so the spectrum disperses along the detector's long axis.
`optika` applies the grating equation automatically during propagation.

In [ ]:
density = 5000 / u.mm
spacing_grating = (1 / density).to(u.nm)
z_grating = 800 * u.mm  # just downstream of the secondary, in the converging beam

# the grating must be large enough to pass the converging beam at z_grating
radius_grating = (
    radius_aperture * (z_parabola_focus - z_grating) / z_parabola_focus + 10 * u.mm
)

grating = optika.surfaces.Surface(
    name="grating",
    aperture=optika.apertures.CircularAperture(radius_grating),
    rulings=optika.rulings.Rulings(
        spacing=optika.rulings.ConstantRulingSpacing(
            constant=spacing_grating,
            normal=na.Cartesian3dVectorArray(1, 0, 0),
        ),
        diffraction_order=1,
    ),
    transformation=na.transformations.Cartesian3dTranslation(z=z_grating),
)

dispersion = ((z_system_focus - z_grating) / spacing_grating).to(u.mm / u.AA)
print(f"grating clear radius = {radius_grating.to(u.mm):.1f}")
print(f"linear dispersion    = {dispersion:.4f}  ({(width_pixel / dispersion).to(u.mAA):.1f}/pixel)")

## Detector

A 2k $\times$ 4k CMOS array with 10 µm pixels.  Real detectors of this kind are
*buttable*, so we make the dispersion axis the long (4k $\approx$ 41 mm) one to
give the spectrum room.  We offset the array along the dispersion direction so
the zero-order image lands near one end and the first-order spectrum (which
disperses toward $-x$) fills the rest of the chip.

In [ ]:
num_pixel = na.Cartesian2dVectorArray(4096, 2048)  # (dispersion, cross-dispersion)
detector_offset = -12 * u.mm

sensor = optika.sensors.ImagingSensor(
    name="sensor",
    width_pixel=width_pixel,
    axis_pixel=na.Cartesian2dVectorArray("detector_x", "detector_y"),
    num_pixel=num_pixel,
    transformation=na.transformations.Cartesian3dTranslation(
        x=detector_offset, z=z_system_focus,
    ),
    is_field_stop=True,
)

## Source

The Sun is effectively at infinity, so the source is specified as a *direction*
rather than a physical aperture: a circular patch of sky $0.5^\circ$ across.
Following the convention used elsewhere in `optika`, giving the aperture radius
as a **dimensionless cosine** marks the surface as being at infinity.

In [ ]:
radius_sun = 0.25 * u.deg  # 0.5 deg full angular diameter

# Place the source well upstream of the telescope.  The paraboloid's outer (most
# upstream) illuminated radius sits near z = -0.3 m on the steep flank, so a
# source at, say, z = -0.2 m would leave those rays travelling *backwards* to
# reach the mirror.  z = -0.7 m keeps every ray moving forward (+z).
source = optika.surfaces.Surface(
    name="solar disk",
    aperture=optika.apertures.CircularAperture(radius=np.cos(radius_sun)),
    transformation=na.transformations.Cartesian3dTranslation(z=-700 * u.mm),
)

## Input rays

The entrance pupil of a Wolter-I telescope is a thin **annulus** — only the
grazing ring is illuminated, the centre is empty.  We therefore build the pupil
grid directly in physical coordinates as a ring at radius $r_0$, with a small
radial half-width set by the mirror's axial length
($\Delta r \approx L_\mathrm{mirror}\tan\alpha / 2$).

In [ ]:
# the pupil grid is the illuminated ring itself: radius r0 +/- radial_halfwidth
# (the mirror extent defined with the paraboloid above).
azimuth = na.linspace(0, 2 * np.pi, axis="pupil_az", num=24, endpoint=False) * u.rad
radius_pupil = na.linspace(
    radius_aperture - radial_halfwidth,
    radius_aperture + radial_halfwidth,
    axis="pupil_r",
    num=5,
)
pupil = na.Cartesian2dVectorArray(
    x=radius_pupil * np.cos(azimuth),
    y=radius_pupil * np.sin(azimuth),
)

The wavelength axis samples the 15–20 Å band, and a small angular field
grid (a few arcminutes) will be used for the spot diagrams.

In [ ]:
wavelength = na.linspace(15, 20, axis="wavelength", num=6) * u.AA

# the detector half-height corresponds to this angular half-field ...
half_field_detector = np.arctan(
    (num_pixel.y * width_pixel / (2 * focal_length)).to_value(u.dimensionless_unscaled)
) * u.rad
print(f"detector half-field = {half_field_detector.to(u.arcmin):.1f}")

# ... but the well-corrected field is only a few arcmin, so sample that for the
# spot diagrams.
field_halfwidth = 5 * u.arcmin
field = na.Cartesian2dVectorLinearSpace(
    start=-field_halfwidth,
    stop=field_halfwidth,
    axis=na.Cartesian2dVectorArray("fx", "fy"),
    num=3,
    centers=True,
)

grid_input = optika.vectors.ObjectVectorArray(
    wavelength=wavelength, field=field, pupil=pupil,
)

## Building the system

`optika.systems.SequentialSystem` assembles the surfaces, the source, the
sensor, and the input grid.

Because the entrance pupil is an annulus specified in **physical** coordinates
(not the normalized $[-1, 1]^2$ square that the automatic backward-raytrace
assumes), we compute the default ray function directly with
`normalized_*=False` and cache it — the same workaround used in the Fresnel
zone-plate tutorial for a non-standard pupil.

In [ ]:
system = optika.systems.SequentialSystem(
    object=source,
    surfaces=[paraboloid, hyperboloid, grating],
    sensor=sensor,
    grid_input=grid_input,
)

rays_default = system.rayfunction(
    wavelength=wavelength,
    field=field,
    pupil=pupil,
    normalized_field=False,
    normalized_pupil=False,
)
system.__dict__["rayfunction_default"] = rays_default
print("object at infinity:", system.object_is_at_infinity)

## Layout

We plot the on-axis ray bundle in the cross-dispersion ($z$–$y$) plane.  The beam
enters from the left, grazes the paraboloid and hyperboloid (the shallow double
fold near the top and bottom of the aperture), passes through the grating, and
converges to the focus.

The mirrors are drawn (black) as their **meridional sag profiles** — the actual
surface cross-section over the illuminated radial band, found from the on-axis
trace.  (`system.plot` instead draws each surface's aperture *rim*; on these
near-cylindrical grazing sags the inner and outer rims fall hundreds of mm apart
in $z$, so the rim outline is not a faithful picture of the optic.)

In [ ]:
raytrace_layout = system.raytrace(
    wavelength=17.5 * u.AA,
    field=na.Cartesian2dVectorArray(0, 0) * u.deg,
    pupil=pupil,
    normalized_field=False,
    normalized_pupil=False,
)


def hit_radius_range(index_surface):
    """Radial extent illuminated on a surface, from the on-axis raytrace."""
    position = raytrace_layout.outputs.position[{system.axis_surface: index_surface}]
    radius = position.xy.length
    return radius.min(), radius.max()


def plot_mirror(ax, surface, radius_range, component, num=51, **kwargs):
    """Draw a mirror's meridional sag profile in the (z, ``component``) plane."""
    radius = na.linspace(radius_range[0], radius_range[1], axis="profile", num=num)
    for sign in [+1, -1]:
        coordinate = sign * radius
        zero = 0 * u.mm
        if component == "y":
            point = na.Cartesian3dVectorArray(x=zero, y=coordinate, z=zero)
        else:
            point = na.Cartesian3dVectorArray(x=coordinate, y=zero, z=zero)
        point = na.Cartesian3dVectorArray(x=point.x, y=point.y, z=surface.sag(point))
        if surface.transformation is not None:
            point = surface.transformation(point)
        na.plt.plot(point, ax=ax, axis="profile", components=("z", component), **kwargs)


# surface order in the trace: object=0, paraboloid=1, hyperboloid=2, grating=3, sensor=4
with astropy.visualization.quantity_support():
    fig, ax = plt.subplots(figsize=(9, 4), constrained_layout=True)
    na.plt.plot(
        raytrace_layout.outputs.position,
        ax=ax,
        axis=system.axis_surface,
        components=("z", "y"),
        color="tab:blue",
        alpha=0.4,
    )
    plot_mirror(ax, paraboloid, hit_radius_range(1), "y", color="black", linewidth=2)
    plot_mirror(ax, hyperboloid, hit_radius_range(2), "y", color="black", linewidth=2)
    ax.set_xlabel(f"$z$ ({u.mm:latex_inline})")
    ax.set_ylabel(f"$y$ ({u.mm:latex_inline})")
    ax.set_title("Cross-dispersion plane: mirror profiles (black) and rays")

### Dispersion layout

The layout above is the cross-dispersion ($z$–$y$) plane, where all diffraction
orders overlap.  To *see* the dispersion we switch to the $z$–$x$ plane and trace
**two orders** through the grating: the un-diffracted **zero order** ($m = 0$,
grey), which forms the direct image at $x \approx 0$, and the **first order**
($m = 1$), traced at 15, 17.5, and 20 Å, which the grating fans out to different
positions on the detector.  The annular pupil appears as two pencils, one from
each side of the ring, and the mirror profiles are again drawn in black.  The
right panel zooms in on the focal plane, where the first-order spectrum is
clearly separated from the zero-order image along the black detector bar.

In [ ]:
import matplotlib as mpl

# meridional (dispersion-plane) slice of the annular pupil: two pencils at +/- r0
azimuth_layout = na.linspace(0, np.pi, axis="pupil_az", num=2) * u.rad
pupil_layout = na.Cartesian2dVectorArray(
    x=radius_pupil * np.cos(azimuth_layout),
    y=radius_pupil * np.sin(azimuth_layout),
)


def raytrace_order(wavelength, order):
    grating_order = optika.surfaces.Surface(
        name="grating",
        aperture=optika.apertures.CircularAperture(radius_grating),
        rulings=optika.rulings.Rulings(
            spacing=optika.rulings.ConstantRulingSpacing(
                constant=spacing_grating, normal=na.Cartesian3dVectorArray(1, 0, 0),
            ),
            diffraction_order=order,
        ),
        transformation=na.transformations.Cartesian3dTranslation(z=z_grating),
    )
    sys_order = optika.systems.SequentialSystem(
        object=source, surfaces=[paraboloid, hyperboloid, grating_order],
        sensor=sensor, grid_input=grid_input,
    )
    return sys_order.raytrace(
        wavelength=wavelength,
        field=na.Cartesian2dVectorArray(0, 0) * u.deg,
        pupil=pupil_layout, normalized_field=False, normalized_pupil=False,
    )


cmap = mpl.colormaps["viridis"]
norm = mpl.colors.Normalize(15, 20)


def draw(ax):
    # zero order is achromatic: straight through to the direct image near x = 0
    na.plt.plot(
        raytrace_order(17.5 * u.AA, order=0).outputs.position, ax=ax,
        axis=system.axis_surface, components=("z", "x"),
        color="0.6", linewidth=0.8,
    )
    # first order, fanned out by wavelength (pass colours as strings: na.plt.plot
    # would mistake an RGBA tuple for a length-4 array)
    for lam_value in [15.0, 17.5, 20.0]:
        na.plt.plot(
            raytrace_order(lam_value * u.AA, order=1).outputs.position, ax=ax,
            axis=system.axis_surface, components=("z", "x"),
            color=mpl.colors.to_hex(cmap(norm(lam_value))), linewidth=0.8,
        )
    # detector extent along the dispersion axis
    x_half = (num_pixel.x * width_pixel / 2).to(u.mm)
    ax.plot(
        u.Quantity([z_system_focus, z_system_focus]).to(u.mm),
        u.Quantity([detector_offset - x_half, detector_offset + x_half]).to(u.mm),
        color="black", linewidth=4, solid_capstyle="butt", zorder=5,
    )


with astropy.visualization.quantity_support():
    fig, (ax, axz) = plt.subplots(
        1, 2, figsize=(12, 4), gridspec_kw=dict(width_ratios=[2, 1]),
        constrained_layout=True,
    )
    draw(ax)
    draw(axz)

    # add the mirror profiles to the full panel (they sit outside the zoom window)
    plot_mirror(ax, paraboloid, hit_radius_range(1), "x", color="black", linewidth=1.5)
    plot_mirror(ax, hyperboloid, hit_radius_range(2), "x", color="black", linewidth=1.5)

    # zoom the right panel on the focal region to separate the orders
    axz.set_xlim(1850 * u.mm, 2100 * u.mm)
    axz.set_ylim(-22 * u.mm, 6 * u.mm)

    sm = mpl.cm.ScalarMappable(norm=norm, cmap=cmap)
    sm.set_array([])
    fig.colorbar(sm, ax=axz, label=f"wavelength ({u.AA:latex_inline})")

    for a in (ax, axz):
        a.set_xlabel(f"$z$ ({u.mm:latex_inline})")
    ax.set_ylabel(f"$x$ ({u.mm:latex_inline})  —  dispersion")
    ax.set_title("Full system: mirrors (black), $m=0$ (grey), $m=1$ (coloured)")
    axz.set_title("Detector region (zoom)")

## Aperture footprints

A useful check is *where* the rays actually land on each optic relative to its
clear aperture.  Tracing the full field and pupil (at 17.5 Å), the blue points
below are the ray intercepts and the black outline is each aperture.  The two
grazing mirrors are illuminated in a thin **annular footprint** that sits
comfortably inside their annular stops (the central hole is the obscuration),
while the still-converging beam under-fills the circular transmission grating.

In [ ]:
# trace the full field + pupil (one wavelength) to map the footprint on each optic
raytrace_footprint = system.raytrace(
    wavelength=17.5 * u.AA,
    field=field,
    pupil=pupil,
    normalized_field=False,
    normalized_pupil=False,
)

# (index in the surface axis, surface, label)
footprint_surfaces = [
    (1, paraboloid, "paraboloid"),
    (2, hyperboloid, "hyperboloid"),
    (3, grating, "grating"),
]

with astropy.visualization.quantity_support():
    fig, axs = plt.subplots(1, 3, figsize=(12, 4.5), constrained_layout=True)
    for (index, surface, name), ax in zip(footprint_surfaces, axs):
        position = raytrace_footprint.outputs.position[{system.axis_surface: index}]
        unvignetted = raytrace_footprint.outputs.unvignetted[{system.axis_surface: index}]
        na.plt.scatter(
            position.x.to(u.mm), position.y.to(u.mm),
            ax=ax, s=2, where=unvignetted, color="tab:blue",
        )
        na.plt.plot(
            surface.aperture.wire(), ax=ax, axis="wire",
            components=("x", "y"), color="black",
        )
        ax.set_aspect("equal")
        ax.set_title(name)
        ax.set_xlabel(f"$x$ ({u.mm:latex_inline})")
    axs[0].set_ylabel(f"$y$ ({u.mm:latex_inline})")

## Dispersion

Tracing on-axis at 15, 17.5, and 20 Å, the first-order images fall in a line
along the dispersion ($x$) axis, evenly spaced at the linear dispersion derived
above ($\approx 0.63$ mm/Å — coarser than a compact layout would give, because
the longer, non-overlapping telescope shortens the grating-to-focus lever arm).

The positions below are in **sensor-local** coordinates.  Because we offset the
detector by $-12$ mm, the un-diffracted **zero order** lands at $x = +12$ mm —
near one end of the 41 mm-long axis — while the 15–20 Å first-order band sits
near $x \approx 0$.  Longer wavelengths disperse further toward $-x$, so the
buttable 4k axis records the zero-order direct image and a broad first-order
spectrum at the same time.

In [ ]:
def centroid_x(wavelength, order=1):
    grating_order = optika.surfaces.Surface(
        name="grating",
        aperture=optika.apertures.CircularAperture(radius_grating),
        rulings=optika.rulings.Rulings(
            spacing=optika.rulings.ConstantRulingSpacing(
                constant=spacing_grating, normal=na.Cartesian3dVectorArray(1, 0, 0),
            ),
            diffraction_order=order,
        ),
        transformation=na.transformations.Cartesian3dTranslation(z=z_grating),
    )
    sys_order = optika.systems.SequentialSystem(
        object=source, surfaces=[paraboloid, hyperboloid, grating_order],
        sensor=sensor, grid_input=grid_input,
    )
    out = sys_order.rayfunction(
        wavelength=wavelength,
        field=na.Cartesian2dVectorArray(0, 0) * u.deg,
        pupil=pupil, normalized_field=False, normalized_pupil=False,
    ).outputs
    x = out.position.x.to(u.mm).ndarray.value
    where = out.unvignetted.ndarray
    return np.mean(x[where])  # mm, unvignetted rays only

for lam in [15, 17.5, 20] * u.AA:
    print(f"  {lam:>5}:  x = {centroid_x(lam):8.3f} mm")

print(f"  zero order:  x = {centroid_x(17.5 * u.AA, order=0):8.3f} mm")

## Spot diagrams

`spot_diagram` shows the geometric PSF at each of the nine field points; the
grating dispersion is removed (each spot is re-centred on its own centroid) and
the points are coloured by wavelength.  The **on-axis** panel (centre) collapses
to a near point — the Wolter-I is aberration-free on-axis — and the six
wavelengths sit almost on top of one another, since the grating adds only a
small blur in the converging beam.

Moving off-axis, the spots quickly grow into the comatic spirals characteristic
of a two-mirror grazing telescope.  Even at the few-arcminute field sampled here
the blur reaches hundreds of microns (tens of pixels), so the off-axis
aberrations — not the grating — set the usable field of view, which for this
simple design is only a couple of arcminutes wide.

In [ ]:
fig, ax = system.spot_diagram()
fig.set_size_inches(8, 8)

## Spatial plate scale

A clean check of the plate scale uses the **cross-dispersion** ($y$) direction,
which the grating leaves untouched.  Displacing the source by an angle
$\theta_y$ shifts the image by $f_\mathrm{eff}\,\theta_y$, so both the effective
focal length and the plate scale $w_\mathrm{pix}/f_\mathrm{eff}$ follow from the
measured shift.

For a *two-mirror* telescope the effective focal length is measured from the rear
principal plane, which sits down among the mirrors — not from the coordinate
origin we used to place the focus.  Now that the primary and secondary are spread
out over $\sim 0.7$ m so they don't overlap, that principal plane sits well
inside the telescope, and the measured $f_\mathrm{eff} \approx 1.78$ m comes out
$\sim 14\%$ short of the 2.06 m back-focal distance.  The plate scale is therefore
$\approx 1.16''$/pixel rather than the $1''$/pixel target — a direct consequence
of the long grazing telescope.  Moving the detector out $\sim 0.3$ m (or
rescaling the whole design) would recover it.

In [ ]:
theta_y = 30 * u.arcsec

out_on = system.rayfunction(
    wavelength=17.5 * u.AA, field=na.Cartesian2dVectorArray(0, 0) * u.deg,
    pupil=pupil, normalized_field=False, normalized_pupil=False,
).outputs
out_off = system.rayfunction(
    wavelength=17.5 * u.AA,
    field=na.Cartesian2dVectorArray(0 * u.deg, theta_y.to(u.deg)),
    pupil=pupil, normalized_field=False, normalized_pupil=False,
).outputs

dy = np.nanmean(out_off.position.y.ndarray.value) - np.nanmean(out_on.position.y.ndarray.value)
dy = np.abs(dy) * out_on.position.y.unit

f_eff = (dy / np.tan(theta_y)).to(u.m)
measured_scale = (theta_y / (dy / width_pixel)).to(u.arcsec)

print(f"image shift          = {dy.to(u.um):.2f}  ({(dy / width_pixel).to_value(u.dimensionless_unscaled):.2f} pixels)")
print(f"effective focal len  = {f_eff:.3f}   (back-focal distance was {focal_length.to(u.m):.3f})")
print(f"measured plate scale = {measured_scale:.3f} / pixel   (target 1.000 arcsec / pixel)")

## Mirror reflectivity at grazing incidence

The geometric model above assumed perfect mirrors.  In reality the grazing-angle
reflectivity sets the throughput.  Using `optika`'s tabulated optical constants
$n = 1 - \delta + i\beta$ (CXRO data), the critical angle for total external
reflection is $\theta_c = \sqrt{2\delta}$, and the single-surface reflectivity
follows from the Fresnel equation

$$R(\theta) = \left|\frac{\sin\theta - \sqrt{n^2 - \cos^2\theta}}
                          {\sin\theta + \sqrt{n^2 - \cos^2\theta}}\right|^2 ,$$

with $\theta$ measured from the *surface*.

In [ ]:
au = optika.chemicals.Chemical("Au")
ir = optika.chemicals.Chemical("Ir")

def reflectivity(chem, graze, wavelength):
    n = chem.n(wavelength)
    s, c = np.sin(graze), np.cos(graze)
    root = np.sqrt(n**2 - c**2)
    return np.abs((s - root) / (s + root)) ** 2

# critical angle of gold across the band
delta_au = 1 - np.real(au.n(wavelength).ndarray)
theta_c = (np.sqrt(2 * delta_au) * u.rad).to(u.deg)
print(f"Au critical angle over 15-20 A: {theta_c.min():.2f} - {theta_c.max():.2f}")
print(f"chosen grazing angle:           {grazing_angle.to(u.deg):.2f}")

In [ ]:
graze = na.linspace(0, 4, axis="graze", num=400) * u.deg

with astropy.visualization.quantity_support():
    fig, (axL, axR) = plt.subplots(1, 2, figsize=(11, 4), constrained_layout=True)

    # reflectivity vs grazing angle at mid-band
    for chem, name, color in [(au, "Au", "tab:orange"), (ir, "Ir", "tab:purple")]:
        R = reflectivity(chem, graze, 17.5 * u.AA)
        na.plt.plot(graze, R, ax=axL, axis="graze", label=name, color=color)
    axL.axvline(grazing_angle.to_value(u.deg), color="k", ls="--", lw=1,
                label=f"design ({grazing_angle.to(u.deg):.2f})")
    axL.set_xlabel("grazing angle (deg)")
    axL.set_ylabel("reflectivity (single bounce)")
    axL.set_title(r"Reflectivity vs angle ($\lambda = 17.5$ Å)")
    axL.legend()

    # two-bounce throughput vs wavelength at the design angle
    wl = na.linspace(15, 20, axis="w", num=101) * u.AA
    for chem, name, color in [(au, "Au", "tab:orange"), (ir, "Ir", "tab:purple")]:
        R = reflectivity(chem, grazing_angle, wl)
        na.plt.plot(wl, R**2, ax=axR, axis="w", label=name, color=color)
    axR.set_xlabel(f"wavelength ({u.AA:latex_inline})")
    axR.set_ylabel("throughput (two bounces)")
    axR.set_title(f"Telescope throughput at {grazing_angle.to(u.deg):.2f}")
    axR.legend()

## Summary

`optika` handled the full grazing-incidence spectrograph without modification:

* **Wolter-I geometry.** A paraboloid (`ParabolicSag`) and a confocal hyperboloid
  (`ConicSag`, $k < -1$) illuminated at large radius give exact on-axis imaging;
  the only subtlety is matching the hyperboloid's far focus to the parabola's
  *true* focus at $F_\mathrm{p} - f_\mathrm{p}$.  The sag solvers were stable even
  at sub-millimetre vertex radii.
* **The mirrors must not overlap in z.** Because the grazing flank is so steep,
  each mirror is axially long, so the secondary has to sit well downstream of the
  primary — otherwise part of the beam cannot reach it travelling forward.  That
  makes the telescope $\sim 0.7$ m long, which is why the effective focal length
  (rearward principal plane) and the grating lever arm — and hence the plate
  scale ($\approx 1.16''$/pix) and dispersion ($\approx 0.63$ mm/Å) — come out a
  bit coarser than a naively compact layout would suggest.
* **Dispersion.** A constant-pitch transmission grating in the converging beam
  produces a linear, well-behaved first-order spectrum with the zero order on the
  same detector.
* **Performance.** The telescope alone is diffraction-limited on-axis
  ($\ll 1$ pixel); the spectral resolution ($R \sim 10^3$) is set mostly by the
  aberration of the plane grating working in the converging beam — a real design
  trade-off that a faster beam would worsen.  A blazed/curved grating or a tilted
  detector would recover more resolution.

Natural next steps: fold the measured reflectivity and grating efficiency into
the ray intensities for an end-to-end throughput estimate, and study the
off-axis field aberrations that ultimately limit the usable field of view.